# 02 — Pilot SAE Training on DINOv2 Activations

Trains a BatchTopK SAE on the cached DINOv2 ViT-B/14 layer-11 activations as a sanity check.

**Pilot config:** 10K images × 256 patches = 2.56M patch tokens, 16× expansion (dict_size=12288), k=192.

In [2]:
# ── Cell 1: Load and prepare data ─────────────────────────────────────────────
import glob
import json
import os
import sys

import torch

# Make sure the repo root is on the path when running from notebooks/
repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

ACTIVATION_DIR = os.path.join(repo_root, "activations", "dinov2_vitb14", "layer_11")

# Load stats
stats_path = os.path.join(ACTIVATION_DIR, "stats.json")
with open(stats_path) as f:
    stats = json.load(f)

mean = torch.tensor(stats["mean"], dtype=torch.float32)  # [768]
std  = float(stats["std"])                                # scalar
print(f"Stats loaded — mean shape: {mean.shape},  std: {std:.4f}")
print(f"Backbone: {stats.get('backbone', 'dinov2_vitb14')}  "
      f"layer: {stats.get('layer', 11)}  "
      f"num_images: {stats.get('num_images')}  "
      f"d_model: {stats.get('d_model')}")

# Load all shards and concatenate
shard_paths = sorted(glob.glob(os.path.join(ACTIVATION_DIR, "shard_*.pt")))
print(f"\nFound {len(shard_paths)} shard(s): {[os.path.basename(p) for p in shard_paths]}")

shards = [torch.load(p, map_location="cpu") for p in shard_paths]
data_4d = torch.cat(shards, dim=0)  # [num_images, 256, 768]
print(f"Concatenated shard shape: {tuple(data_4d.shape)}  (num_images × patches × d_model)")

# Reshape to flat patch tokens
num_images, num_patches, d_model = data_4d.shape
data = data_4d.reshape(num_images * num_patches, d_model).float()  # [N*256, 768]
print(f"Flat patch tokens shape:  {tuple(data.shape)}")

# Normalize using stats.json
data = (data - mean) / std
print(f"Normalized — sample mean: {data.mean():.4f}  std: {data.std():.4f}  (should be ≈0, ≈1)")

# Shuffle
perm = torch.randperm(data.shape[0])
data = data[perm]

print(f"\nTotal training samples: {data.shape[0]:,}  shape: {tuple(data.shape)}")

Stats loaded — mean shape: torch.Size([768]),  std: 1.8913
Backbone: dinov2_vitb14  layer: 11  num_images: 10000  d_model: 768

Found 2 shard(s): ['shard_000.pt', 'shard_001.pt']
Concatenated shard shape: (10000, 256, 768)  (num_images × patches × d_model)
Flat patch tokens shape:  (2560000, 768)
Normalized — sample mean: 0.0000  std: 1.0000  (should be ≈0, ≈1)

Total training samples: 2,560,000  shape: (2560000, 768)


In [3]:
# ── Cell 2: Configure and instantiate BatchTopK SAE ───────────────────────────
from overcomplete import BatchTopKSAE

from src.training.overcomplete_config import make_sae_config

D_MODEL          = 768
EXPANSION_FACTOR = 16
DICT_SIZE        = D_MODEL * EXPANSION_FACTOR  # 12288
K                = 192   # active features per batch step
THRESHOLD_MOM    = 0.9

device = "cuda" if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else "cpu"
print(f"Device: {device}")

config = make_sae_config(
    d_model=D_MODEL,
    expansion_factor=EXPANSION_FACTOR,
    k=K,
    architecture="batchtopk",
)
print(f"Config: {config}")

# Instantiate via overcomplete and call .tied() to use dictionary^T as the encoder.
# The default MLPEncoder uses BatchNorm1d(12288) which is very slow on CPU/MPS.
# Tied mode is standard SAE practice and avoids this overhead entirely.
sae = BatchTopKSAE(
    input_shape=D_MODEL,
    nb_concepts=DICT_SIZE,
    top_k=K,
    threshold_momentum=THRESHOLD_MOM,
    device=device,
).tied()

num_params = sum(p.numel() for p in sae.parameters())
print(f"\nBatchTopKSAE instantiated (tied encoder):")
print(f"  input_shape:  {D_MODEL}")
print(f"  nb_concepts:  {DICT_SIZE}  ({EXPANSION_FACTOR}× expansion)")
print(f"  top_k:        {K}")
print(f"  parameters:   {num_params:,}")

Device: mps
Config: {'architecture': 'batchtopk', 'd_model': 768, 'expansion_factor': 16, 'dict_size': 12288, 'k': 192, 'constructor_kwargs': {'input_shape': 768, 'nb_concepts': 12288, 'top_k': 192, 'threshold_momentum': 0.9}}

BatchTopKSAE instantiated (tied encoder):
  input_shape:  768
  nb_concepts:  12288  (16× expansion)
  top_k:        192
  parameters:   9,437,184


In [5]:
# ── Cell 3: Train the SAE ─────────────────────────────────────────────────────
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

BATCH_SIZE  = 4096
LR          = 3e-4
NUM_EPOCHS  = 2   # ~10–30 min on a single GPU for 2.56M samples
LOG_EVERY   = 50   # log loss every N steps

dataset = TensorDataset(data)
loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=False, num_workers=0)

optimizer = torch.optim.Adam(sae.parameters(), lr=LR)

steps_per_epoch = len(loader)
total_steps     = NUM_EPOCHS * steps_per_epoch
print(f"Training: {NUM_EPOCHS} epochs × {steps_per_epoch} steps = {total_steps} total steps")
print(f"Batch size: {BATCH_SIZE}  |  LR: {LR}")

training_log = []  # [(step, loss), ...]
global_step  = 0

sae.train()
for epoch in range(NUM_EPOCHS):
    epoch_loss = 0.0
    for (batch,) in tqdm(loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}", leave=False):
        batch = batch.to(device)
        optimizer.zero_grad()

        # forward() returns a plain tuple: (pre_codes, codes, x_reconstructed)
        _pre_codes, codes, x_hat = sae(batch)
        loss = (batch - x_hat).square().mean()
        loss.backward()
        optimizer.step()

        loss_val    = float(loss.item())
        epoch_loss += loss_val
        global_step += 1

        if global_step % LOG_EVERY == 0:
            training_log.append({"step": global_step, "loss": loss_val})

    avg = epoch_loss / steps_per_epoch
    print(f"Epoch {epoch+1:>2}/{NUM_EPOCHS}  |  step {global_step:>6}  |  avg loss: {avg:.4f}")

print("\nTraining complete.")

Training: 2 epochs × 625 steps = 1250 total steps
Batch size: 4096  |  LR: 0.0003


Epoch 1/2:   0%|          | 0/625 [00:00<?, ?it/s]

Epoch  1/2  |  step    625  |  avg loss: 0.9355


Epoch 2/2:   0%|          | 0/625 [00:00<?, ?it/s]

Epoch  2/2  |  step   1250  |  avg loss: 0.9355

Training complete.


In [ ]:
# ── Cell 4: Evaluate basic metrics ────────────────────────────────────────────
# Use a held-out subset (last 50k samples after shuffle) for FVU / L0 / dead features.
# For dead-feature counting we pass the full dataset.

EVAL_BATCH = 4096

sae.eval()

# ── 4a. FVU and L0 on held-out 50k patch tokens ──
eval_data = data[-50_000:]

all_x, all_x_hat, all_codes = [], [], []
with torch.no_grad():
    for start in range(0, len(eval_data), EVAL_BATCH):
        batch = eval_data[start : start + EVAL_BATCH].to(device)
        # forward() returns a plain tuple: (pre_codes, codes, x_reconstructed)
        _pre_codes, codes, x_hat = sae(batch)
        all_x.append(batch.cpu())
        all_x_hat.append(x_hat.cpu())
        all_codes.append(codes.cpu())

x     = torch.cat(all_x, dim=0)
x_hat = torch.cat(all_x_hat, dim=0)
codes = torch.cat(all_codes, dim=0)

# FVU = Var(x - x_hat) / Var(x)
fvu = float((x - x_hat).var() / x.var())

# L0 = mean number of non-zero activations per input
l0 = float((codes != 0).float().sum(dim=1).mean())

# ── 4b. Dead features on full dataset ──
all_full_codes = []
with torch.no_grad():
    for start in range(0, len(data), EVAL_BATCH):
        batch = data[start : start + EVAL_BATCH].to(device)
        _pre_codes, codes_full, _x_hat = sae(batch)
        all_full_codes.append(codes_full.cpu())

full_codes  = torch.cat(all_full_codes, dim=0)
ever_active = (full_codes != 0).any(dim=0)          # [dict_size]
dead_count  = int((~ever_active).sum())
dead_pct    = 100.0 * dead_count / DICT_SIZE

print("=" * 50)
print("Evaluation Metrics")
print("=" * 50)
print(f"  FVU (Fraction of Variance Unexplained): {fvu:.4f}   (target < 0.10)")
print(f"  L0  (avg active features / input):      {l0:.1f}    (target ≈ {K})")
print(f"  Dead features: {dead_count} / {DICT_SIZE}  ({dead_pct:.1f}%)  (target < 10%)")
print("=" * 50)

In [ ]:
# ── Cell 5: Save checkpoint ────────────────────────────────────────────────────
import yaml

CHECKPOINT_DIR = os.path.join(repo_root, "checkpoints", "dinov2_vitb14", "batchtopk_16x_k192")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Save SAE weights
sae_path = os.path.join(CHECKPOINT_DIR, "sae.pt")
torch.save(sae.state_dict(), sae_path)
print(f"Saved SAE weights  → {sae_path}")

# Save config
cfg = {
    "backbone":         "dinov2_vitb14",
    "architecture":     "batchtopk",
    "d_model":          D_MODEL,
    "expansion_factor": EXPANSION_FACTOR,
    "dict_size":        DICT_SIZE,
    "k":                K,
    "threshold_momentum": THRESHOLD_MOM,
    "lr":               LR,
    "batch_size":       BATCH_SIZE,
    "num_epochs":       NUM_EPOCHS,
    "total_steps":      global_step,
    "activation_dir":   ACTIVATION_DIR,
}
config_path = os.path.join(CHECKPOINT_DIR, "config.yaml")
with open(config_path, "w") as f:
    yaml.dump(cfg, f, default_flow_style=False)
print(f"Saved config       → {config_path}")

# Save training log + final metrics
log_path = os.path.join(CHECKPOINT_DIR, "training_log.json")
with open(log_path, "w") as f:
    json.dump(
        {
            "training_loss": training_log,
            "final_metrics": {
                "fvu":          fvu,
                "l0":           l0,
                "dead_features": dead_count,
                "dead_pct":     dead_pct,
            },
        },
        f,
        indent=2,
    )
print(f"Saved training log → {log_path}")
print("\nAll done!")